In [ ]:
# ==============================================================================
# 📓 Advanced RAG Cookbook: 01_data_loading.ipynb
# ==============================================================================
# الهدف: فهم وإتقان استيراد البيانات من مختلف المصادر وتحويلها إلى كائنات Document
# تحوي النص الصافي مع البيانات الوصفية (Metadata).
# ==============================================================================

# ------------------------------------------------------------------------------
# 0. التثبيتات والإعدادات الأولية (Requirements Setup)
# ------------------------------------------------------------------------------
# قم بفك التعليق عن السطر التالي وتفيذه مرة واحدة داخل الـ Notebook
# !pip install langchain-community pypdf unstructured sentence-transformers bs4

import os
from langchain_community.document_loaders import (
    TextLoader,
    WebBaseLoader,
    PyPDFLoader,
    UnstructuredPDFLoader
)

print("✅ تم استيراد المكتبات بنجاح!")


# ==============================================================================
# 1. Standard Text Loading (TextLoader)
# ==============================================================================

📌 الشرح النظري:
أبسط وأسرع طريقة لتحميل البيانات النصية غير المنسقة (.txt, .md).
تقرأ الملف بالكامل ككتلة نصية واحدة وتضع البيانات الوصفية الأساسية مثل مسار الملف.

💡 المميزات:
- خفيف جداً ولا يستغرق أي وقت في التشغيل.
- لا يحتاج لمكتبات معقدة أو نماذج خارجية.

⚠️ العيوب:
- لا يتعامل إلا مع النصوص السادة (Plain Text).
- يفقد أي تنسيقات معقدة أو هيكلية مستندات.



In [ ]:

# إنشاء ملف نصي تجريبي للمعاينة
sample_text_path = "sample_notes.txt"
with open(sample_text_path, "w", encoding="utf-8") as f:
    f.write("Retrieval-Augmented Generation (RAG) improves LLM responses using external knowledge sources.")

# 1. إعداد الـ Loader
text_loader = TextLoader(sample_text_path, encoding="utf-8")

# 2. تحميل البيانات
text_docs = text_loader.load()

# 3. طباعة النتائج
print("--- 1. TextLoader Output ---")
print(f"📄 عدد المستندات: {len(text_docs)}")
print(f"📝 النص المستخرج: {text_docs[0].page_content}")
print(f"🏷️ البيانات الوصفية (Metadata): {text_docs[0].metadata}\n")

# ==============================================================================
# 2. Web Scraping & HTML Loading (WebBaseLoader)
# ==============================================================================

📌 الشرح النظري:

تُستخدم لاستخراج النصوص من صفحات الويب والمقالات عبر الإنترنت، حيث تقوم بتنزيل 

كود الـ HTML وتصفية الهاشتاج والإعلانات وعناصر التنسيق للوصول للنص الصافي 

مع حفظ رابط الصفحة كـ Metadata.

💡 المميزات:
- أداة ممتازة لبناء أنظمة RAG تضمن التحديث الفوري من الإنترنت.
- تجلب العناوين والروابط بشكل تلقائي وتضعها في الـ Metadata.

⚠️ العيوب:
- تعتمد على استقرار موقع الويب والاتصال بالشبكة.
- بعض المواقع المعقدة (SPA) تستلزم أدوات إضافية مثل Selenium أو Playwright.



In [ ]:

# 1. إعداد الـ Loader برابط مقال
url = "https://en.wikipedia.org/wiki/Generative_artificial_intelligence"
web_loader = WebBaseLoader(url)

# 2. تحميل المحتوى
web_docs = web_loader.load()

# 3. طباعة النتائج
print("--- 2. WebBaseLoader Output ---")
print(f"🌐 رابط المصدر: {web_docs[0].metadata['source']}")
print(f"📝 معاينة النص (أول 200 حرف): {web_docs[0].page_content[:200].strip()}...\n")

# ==============================================================================
# 3. Standard PDF Extraction (PyPDFLoader)
# ==============================================================================

📌 الشرح النظري:  
الطريقة القياسية المتبعة مع ملفات الـ PDF الشائعة. تقوم بفصل كل صفحة في الـ PDF 

إلى كائن Document مستقل وتلقائياً تضيف رقم الصفحة (page) في الـ Metadata،
 
مما يسهل عملية إظهار المصدر (Citations) للمستخدم لاحقاً.

💡 المميزات:
- سرعة فائقة في معالجة الملفات الكبيرة.
- فصل الصفحات تلقائياً يحافظ على دقة إسناد المراجع.

⚠️ العيوب:
- تفشل في قراءة الجداول المعقدة وتدمج الأعمدة بشكل عشوائي.
- لا تستطيع قراءة الـ PDFs المكسورة أو الممسوحة ضوئياً (Scanned/Images).



In [ ]:

# إنشاء ملف PDF تجريبي مبسط للتشغيل المباشر
# ملاحظة: يمكنك تغيير المسار لملف PDF حقيقي لديك
pdf_path = "sample_paper.pdf"

# سنحاول تحميل ملف موجود أو إعطاء تنبيه
if os.path.exists(pdf_path):
    pdf_loader = PyPDFLoader(pdf_path)
    pdf_docs = pdf_loader.load()

    print("--- 3. PyPDFLoader Output ---")
    print(f"📄 إجمالي الصفحات: {len(pdf_docs)}")
    print(f"🏷️ بيانات الصفحة الأولى: {pdf_docs[0].metadata}")
    print(f"📝 محتوى الصفحة الأولى: {pdf_docs[0].page_content[:200]}...\n")
else:
    print("--- 3. PyPDFLoader Output ---")
    print(f"⚠️ الملف {pdf_path} غير موجود. يرجى إرفاق ملف PDF واختبار الكود عليه.\n")

# ==============================================================================
# 4. Advanced Structural Parsing (UnstructuredPDFLoader)
# ==============================================================================

```python
📌 الشرح النظري:
التقنية الاحترافية المستخدمة في الـ Advanced RAG لملفات المستندات المعقدة 
(التقارير المالية، الأوراق البحثية). لا تكتفي بقراءة النص، بل تعتمد على أدوات 
فهم الهيكل (Layout Detection) للتمييز بين النص الأصلي، الجداول، والعناوين الفرعية، 
وتحول الجداول إلى صيغة Markdown ليتمكن الـ LLM من فهم العلاقات الرقمية داخلها.

💡 المميزات:
- أفضل دقة على الإطلاق في استخراج الجداول وتحويلها لبيانات يستوعبها النموذج.
- تميز بين الهوامش، العناوين، والنص الأساسي وتمنع التداخل الدلالي.

⚠️ العيوب:
- أبطأ بكثير من الـ Loaders العادية.
- تستهلك موارد جهاز عالية (CPU/GPU) أو تحتاج لـ API خارجي.
```

In [ ]:

complex_pdf_path = "complex_report.pdf"

print("--- 4. UnstructuredPDFLoader Output ---")
if os.path.exists(complex_pdf_path):
    # إعداد الـ Loader مع الحفاظ على التناسق والتركيب Structural Elements
    advanced_loader = UnstructuredPDFLoader(
        file_path=complex_pdf_path,
        mode="elements", # يستخرج العناصر مقسمة (Titles, NarrativeText, Tables)
        strategy="hi_res" # استخدام الدقة العالية لتحليل الهيكل والجداول
    )
    advanced_docs = advanced_loader.load()

    print(f"🧩 إجمالي العناصر الهيكلية: {len(advanced_docs)}")
    print(f"🏷️ نوع العنصر الأول: {advanced_docs[0].metadata.get('category')}")
    print(f"📝 محتوى العنصر الأول: {advanced_docs[0].page_content}\n")
else:
    print(f"⚠️ الملف {complex_pdf_path} غير موجود. يلزم وجود ملف معقد لتجربة هذه الميزة.\n")

# ==============================================================================
# 4.1 Optional: Advanced Cloud Parsing (LlamaParse)
# ==============================================================================
"""
📌 الش الشرح النظري:
أداة سحابية متقدمة مبنية على نماذج الرؤية (Vision Models) لقراءة ملفات PDF 
المعقدة جداً (مثل التقارير المالية والتحليلات البيانية). تحول الجداول والرسوم 
إلى صيغة Markdown فائقة الدقة.

💡 المميزات:
- أداء استثنائي في تحويل الجداول الملتوية والمربكة إلى Markdown Tables.
- تفهم التخطيطات متعددة الأعمدة (Multi-column) ببراعة.

⚠️ العيوب:
- تتطلب حظر بيانات (تتطلب الاتصال بـ Cloud API وليس محلياً).
- تحتاج API Key مجاني من موقع cloud.llamaindex.ai مع حد مجاني (1000 صفحة/شهرياً).

In [ ]:
# يتطلب تثبيت: pip install llama-parse
import os

# ضع الـ API Key الخاص بك هنا لتجربة الخلية
LLAMA_CLOUD_KEY = "your_llama_cloud_api_key_here"

if LLAMA_CLOUD_KEY != "your_llama_cloud_api_key_here":
    from llama_parse import LlamaParse
    
    os.environ["LLAMA_CLOUD_API_KEY"] = LLAMA_CLOUD_KEY
    
    parser = LlamaParse(
        result_type="markdown", # تحويل المخرجات إلى ماركداون
        verbose=True
    )
    
    # تجربة تحميل ملف
    # documents = parser.load_data("financial_report.pdf")
    # print(documents[0].text[:300])
else:
    print("⚠️ لتشغيل LlamaParse، يرجى التسجيل في cloud.llamaindex.ai ووالمتابعة بإضافة الـ API Key.")

# ==============================================================================
# 📝 الخلاصة للـ Notebook (Takeaways)
# ==============================================================================

1. القاعدة الذهبية: اختر الـ Loader بناءً على درجة تعقيد مستنداتك، وليس بناءً على الأسهل.
2. الـ Metadata هي المحرك الأساسي: دائماً تأكد من وجود اسم المصدر (source) 
   ورقم الصفحة/الرابط لتمكين ميزات الـ Filtering والـ Citations في تطبيقك المتقدم.
3. في بيئة الإنتاج (Production): بالنسبة للتقارير والأوراق البحثية، 
   يوصى بالاعتماد على Unstructured أو خدمات مثل LlamaParse.

print("✅ اكتمل تشغيل وقراءة نوت بوك Data Loading!")